# Notebook 7: MLS Salary Analysis (Real Data)
**Source:** MLSPA Spring 2026 Salary Guide — 916 players, 31 clubs, as of April 14, 2026
**File:** `data/raw/professional/mlspa_2026.csv` (downloaded directly from MLSPA S3)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

df = pd.read_csv('../data/processed/mls_player_salaries_2026.csv')
print(f'{len(df):,} players across {df["club"].nunique()} clubs')
print(df.dtypes)

## 1. League-Wide Salary Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Histogram (log scale) ---
axes[0].hist(df['base_salary'], bins=60, color='#2980b9', alpha=0.8, edgecolor='white')
axes[0].set_xscale('log')
axes[0].axvline(df['base_salary'].median(), color='red', linestyle='--',
                label=f'Median: ${df["base_salary"].median():,.0f}')
axes[0].axvline(df['base_salary'].mean(), color='orange', linestyle='--',
                label=f'Mean: ${df["base_salary"].mean():,.0f}')
axes[0].set_title('MLS Base Salary Distribution (Log Scale)\n2026 Season — 916 Players', fontsize=12)
axes[0].set_xlabel('Base Salary (USD, log scale)')
axes[0].set_ylabel('Number of Players')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'${v/1e6:.1f}M' if v >= 1e6 else f'${v/1e3:.0f}k'))
axes[0].legend()

# --- Salary tier breakdown ---
bins = [0, 88025, 113400, 250000, 500000, 1000000, 3000000, 30000000]
labels = ['Min\n$88k', 'Reserve\n$113k', '$114k–\n$250k', '$251k–\n$500k',
          '$501k–\n$1M', '$1M–\n$3M', '$3M+']
df['tier'] = pd.cut(df['base_salary'], bins=bins, labels=labels)
counts = df['tier'].value_counts().sort_index()
colors = ['#e74c3c','#e67e22','#f1c40f','#2ecc71','#3498db','#9b59b6','#1abc9c']
bars = axes[1].bar(counts.index, counts.values, color=colors, alpha=0.85, edgecolor='white')
axes[1].set_title('Players by Salary Tier\n2026 MLS Season', fontsize=12)
axes[1].set_ylabel('Number of Players')
for bar, val in zip(bars, counts.values):
    pct = val / len(df) * 100
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
                f'{val}\n({pct:.0f}%)', ha='center', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.savefig('../data/analysis/mls_salary_distribution_2026.png', dpi=150)
plt.show()

print(f"\nKey stats (2026 MLSPA)")
for label, val in [
    ('Minimum salary',     df['base_salary'].min()),
    ('25th percentile',    df['base_salary'].quantile(0.25)),
    ('Median',             df['base_salary'].median()),
    ('Mean',               df['base_salary'].mean()),
    ('75th percentile',    df['base_salary'].quantile(0.75)),
    ('90th percentile',    df['base_salary'].quantile(0.90)),
    ('Maximum (Messi)',    df['base_salary'].max()),
    ('Total league payroll', df['base_salary'].sum()),
]:
    print(f'  {label:<25}: ${val:>15,.0f}')

## 2. Club Payroll Comparison

In [ ]:
club = df.groupby('club').agg(
    players=('base_salary','count'),
    total_payroll=('base_salary','sum'),
    median_salary=('base_salary','median'),
    max_salary=('base_salary','max')
).sort_values('total_payroll', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Total payroll
colors_c = ['#e74c3c' if c == 'Inter Miami' else '#3498db' for c in club.index]
axes[0].barh(club.index[::-1], club['total_payroll'][::-1] / 1e6, color=colors_c[::-1], alpha=0.85)
axes[0].set_xlabel('Total Payroll (USD millions)')
axes[0].set_title('MLS Club Total Payroll (2026)\nRed = Inter Miami (Messi effect)', fontsize=12)
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'${v:.0f}M'))

# Median salary (better fairness measure)
club_med = club.sort_values('median_salary', ascending=False)
axes[1].barh(club_med.index[::-1], club_med['median_salary'][::-1] / 1e3, color='#27ae60', alpha=0.85)
axes[1].set_xlabel('Median Player Salary (USD thousands)')
axes[1].set_title('MLS Club Median Player Salary (2026)\n(better indicator of roster depth investment)', fontsize=12)
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'${v:.0f}k'))

plt.tight_layout()
plt.savefig('../data/analysis/mls_club_payroll_2026.png', dpi=150)
plt.show()

## 3. Reality Check: What a Pro Career Actually Pays

In [ ]:
# Pull in cost data for comparison
costs = pd.read_csv('../data/processed/costs.csv')

# Career cost Tier 3 (ECNL), 7 years
tier3_career_cost = costs[costs['tier'] == 3]['total_mid'].mean() * 7
tier3_career_high = costs[costs['tier'] == 3]['total_high'].mean() * 7

# MLS career scenarios (Watanabe 2010: avg 2.4yr initial, ~6.6yr if reach yr4)
scenarios = {
    'At MLS minimum\n(2.4yr career)':       88025 * 2.4,
    'At MLS median\n(2.4yr career)':         325000 * 2.4,
    'At MLS median\n(6.6yr career)':         325000 * 6.6,
    'At MLS mean\n(6.6yr career)':           604606 * 6.6,
    'At $1M (top 13%)\n(6.6yr career)':      1000000 * 6.6,
}

fig, ax = plt.subplots(figsize=(10, 5))
sce_labels = list(scenarios.keys())
sce_values = list(scenarios.values())
bar_colors = ['#e74c3c' if v < tier3_career_cost else '#27ae60' for v in sce_values]

bars = ax.barh(sce_labels, [v/1e3 for v in sce_values], color=bar_colors, alpha=0.85)
ax.axvline(tier3_career_cost/1e3, color='black', linestyle='--', linewidth=2,
           label=f'Tier 3 career investment (mid): ${tier3_career_cost:,.0f}')
ax.axvline(tier3_career_high/1e3, color='gray', linestyle=':', linewidth=1.5,
           label=f'Tier 3 career investment (high): ${tier3_career_high:,.0f}')

ax.set_xlabel('Total Career Earnings (USD thousands)')
ax.set_title('MLS Career Earnings vs Youth Soccer Investment\n(Red = does NOT recoup Tier 3 cost; Green = does)', fontsize=12)
ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'${v:.0f}k'))

for bar, val in zip(bars, sce_values):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
            f'${val/1e3:,.0f}k', va='center', fontsize=8)

plt.tight_layout()
plt.savefig('../data/analysis/mls_earnings_vs_investment.png', dpi=150)
plt.show()

print(f'\nTier 3 career investment (7yr mid): ${tier3_career_cost:,.0f}')
print(f'Break-even requires MLS salary of:  ${tier3_career_cost/2.4:,.0f}/yr (avg 2.4yr career)')
print(f'That is the {(df["base_salary"] >= tier3_career_cost/2.4).mean()*100:.0f}th percentile of MLS salaries')

## 4. Top Earners — The DP Effect

In [ ]:
top20 = df.nlargest(20, 'base_salary')[['first_name','last_name','club','position','base_salary','guaranteed_comp']]
top20['name'] = top20['first_name'] + ' ' + top20['last_name']

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(top20['name'][::-1], top20['base_salary'][::-1] / 1e6, color='#e74c3c', alpha=0.8, label='Base Salary')
ax.barh(top20['name'][::-1], (top20['guaranteed_comp'][::-1] - top20['base_salary'][::-1]) / 1e6,
        left=top20['base_salary'][::-1] / 1e6, color='#c0392b', alpha=0.5, label='Bonus/Incentives')

ax.set_xlabel('Salary (USD millions)')
ax.set_title('Top 20 MLS Earners — 2026 Season\n(Base + Guaranteed Compensation)', fontsize=12)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'${v:.0f}M'))
ax.legend()

# Add league median reference
ax.axvline(df['base_salary'].median()/1e6, color='blue', linestyle='--', alpha=0.5,
           label=f'League median: ${df["base_salary"].median()/1e3:.0f}k')

plt.tight_layout()
plt.savefig('../data/analysis/mls_top20_earners_2026.png', dpi=150)
plt.show()

# Key insight: DP skew
dp_threshold = 683750  # 2026 DP budget charge
dp_players = df[df['base_salary'] > dp_threshold]
non_dp = df[df['base_salary'] <= dp_threshold]
print(f'\nDesignated Player threshold (approx): ${dp_threshold:,}')
print(f'Players above DP threshold: {len(dp_players)} ({len(dp_players)/len(df)*100:.1f}%)')
print(f'Their share of total payroll: {dp_players["base_salary"].sum()/df["base_salary"].sum()*100:.1f}%')
print(f'\nNon-DP median: ${non_dp["base_salary"].median():,.0f}')
print(f'Non-DP mean:   ${non_dp["base_salary"].mean():,.0f}')

## 5. Key Takeaway: What Is the Realistic MLS Probability Worth?

In [ ]:
# Probability of reaching MLS: ~600 US slots out of 4M youth players = 0.015%
# But we now have real salary data

prob_mls = 600 / 4_000_000  # US roster spots / total youth players

# Expected value of MLS career from each salary tier
tier3_annual_cost = costs[costs['tier'] == 3]['total_mid'].mean()

mls_career_scenarios = [
    ('Minimum earner (17% of MLS)', 88025,   2.4, 0.17),
    ('Median earner (50% of MLS)',  325000,  2.4, 0.50),
    ('Top 25% earner',              656750,  4.0, 0.25),
    ('Top 13% ($1M+)',              1000000, 6.6, 0.13),
]

print('Expected value of MLS career path (Tier 3 player):')
print(f'  Probability of reaching MLS as US player: {prob_mls*100:.3f}%\n')

total_ev = 0
for label, salary, years, prob_given_mls in mls_career_scenarios:
    earnings = salary * years
    weighted_ev = prob_mls * prob_given_mls * earnings
    total_ev += weighted_ev
    print(f'  {label}')
    print(f'    Earnings: ${earnings:,.0f}  |  Prob-weighted EV: ${weighted_ev:,.0f}')

print(f'\nTotal expected value (MLS career): ${total_ev:,.0f}')
print(f'Annual Tier 3 cost (mid):          ${tier3_annual_cost:,.0f}/yr')
print(f'Tier 3 career cost (7yr):          ${tier3_annual_cost*7:,.0f}')
print(f'\nConclusion: The expected monetary return from MLS alone')
print(f'is ${total_ev:,.0f} vs a Tier 3 investment of ${tier3_annual_cost*7:,.0f}')
print(f'That is {total_ev/(tier3_annual_cost*7)*100:.1f}% return on investment from pro soccer alone.')
print(f'The scholarship pathway is separately more likely and valuable.')